# NER propre à Djibouti — Baselines (Mako)

Master 2 IAMD, Université de Djibouti — module TALN.

**Question :** peut-on extraire les institutions, lieux, personnes et dates dans des textes français de Djibouti ?

**Plan du notebook**
1. Charger les données et les annotations  ← *cette section*
2. Baseline 1 : spaCy `fr_core_news_sm` seul
3. Baseline 2 : spaCy + `EntityRuler` (lexique local + dates)
4. Comparaison des deux baselines (sur le test)
5. Analyse de 10 erreurs
6. Démo avec `displacy`

## Section 1 — Charger les données

**Ce qu'on fait :**
- on lit les deux fichiers : les textes (`corpus_ner.csv`) et les annotations (`annotations.csv`) ;
- pour chaque segment, on regroupe ses entités sous forme de triplets `(debut, fin, type)` ;
- on sépare le **dev** (pour écrire les règles) et le **test** (pour mesurer, sans y toucher).

**Pourquoi des triplets ?** C'est le format attendu par `src/evaluate.py`. Exemple : dans `"Il est à Djibouti."`, l'entité `Djibouti` donne le triplet `(10, 18, "LOC")`.

In [1]:
from pathlib import Path
from collections import Counter

import pandas as pd

# Le notebook est dans notebooks/ : les données sont un dossier plus haut.
RACINE = Path("..").resolve()
DATA = RACINE / "data"

corpus = pd.read_csv(DATA / "corpus_ner.csv", encoding="utf-8")
annotations = pd.read_csv(DATA / "annotations.csv", encoding="utf-8", keep_default_na=False)

print("Segments dans le corpus :", len(corpus))
print("Lignes d'annotation     :", len(annotations))
print("Colonnes du corpus      :", list(corpus.columns))
print("Colonnes des annotations:", list(annotations.columns))

Segments dans le corpus : 126
Lignes d'annotation     : 84
Colonnes du corpus      : ['id', 'titre', 'rubrique', 'url', 'date', 'fr', 'split']
Colonnes des annotations: ['id_segment', 'split', 'texte', 'entite', 'type', 'debut', 'fin', 'a_verifier', 'remarque']


**Attention : segments sans entité.** Le fichier d'annotations n'a aucune ligne pour un segment sans entité. Mais ces segments comptent dans l'évaluation : si le modèle y invente une entité, c'est une erreur (fausse). On leur donne donc une liste **vide**.

In [2]:
# gold[id_segment] = liste de triplets (debut, fin, type)
gold = {id_seg: [] for id_seg in corpus["id"]}

for ligne in annotations.itertuples():
    gold[ligne.id_segment].append((int(ligne.debut), int(ligne.fin), ligne.type))

# On trie pour avoir un ordre stable
for id_seg in gold:
    gold[id_seg].sort()

# Vérification : tous les segments annotés existent dans le corpus ?
inconnus = set(annotations["id_segment"]) - set(corpus["id"])
print("Segments annotés absents du corpus :", len(inconnus))

n_vides = sum(1 for ents in gold.values() if not ents)
print("Segments avec au moins une entité  :", len(gold) - n_vides)
print("Segments sans entité (liste vide)  :", n_vides)

Segments annotés absents du corpus : 0
Segments avec au moins une entité  : 48
Segments sans entité (liste vide)  : 78


**Dev et test.** On crée, pour chaque split, trois listes de même longueur et dans le même ordre : les ids, les textes, et les entités de la vérité terrain.

In [3]:
def charger_split(nom):
    sous = corpus[corpus["split"] == nom]
    ids = list(sous["id"])
    textes = list(sous["fr"])
    verite = [gold[i] for i in ids]
    return ids, textes, verite

ids_dev, textes_dev, gold_dev = charger_split("dev")
ids_test, textes_test, gold_test = charger_split("test")

print("dev  :", len(ids_dev), "segments")
print("test :", len(ids_test), "segments")

dev  : 84 segments
test : 42 segments


**Comptes réels par split et par type.**

In [4]:
TYPES = ["ORG", "LOC", "PER", "DATE", "FORMATION"]

def compter(gold_liste):
    c = Counter(t for ents in gold_liste for (_, _, t) in ents)
    return {t: c.get(t, 0) for t in TYPES}

tableau = pd.DataFrame({"dev": compter(gold_dev), "test": compter(gold_test)})
tableau["total"] = tableau["dev"] + tableau["test"]
tableau.loc["TOTAL"] = tableau.sum()
tableau

,dev,test,total
ORG,17,13,30
LOC,16,4,20
PER,14,7,21
DATE,10,3,13
FORMATION,0,0,0
TOTAL,57,27,84


**Lecture :**
- le **test est petit** : peu d'entités par type, donc un seul changement fait bouger le score de plusieurs points ;
- **FORMATION = 0** : aucun diplôme dans ce corpus, on n'en a pas inventé.

In [5]:
# Un exemple concret : un segment, ses triplets, et le texte coupé avec ces positions
exemple = "A03_S002"
texte = corpus.loc[corpus["id"] == exemple, "fr"].iloc[0]
print(texte, "\n")
for debut, fin, type_ in gold[exemple]:
    print(f"({debut}, {fin}, {type_!r:6})  ->  {texte[debut:fin]!r}")

Réunis en Assemblée générale extraordinaire au siège de la Fédération, ils ont voté à l'unanimité la révocation du Comité directeur et de son président, Souleiman Hassan Waberi. 

(115, 131, 'ORG' )  ->  'Comité directeur'
(153, 176, 'PER' )  ->  'Souleiman Hassan Waberi'


In [6]:
# Contrôle final : pour TOUTES les annotations, le texte coupé doit être l'entité notée
ok = all(
    corpus.loc[corpus["id"] == l.id_segment, "fr"].iloc[0][l.debut:l.fin] == l.entite
    for l in annotations.itertuples()
)
print("Toutes les positions correspondent au texte :", ok)

Toutes les positions correspondent au texte : True


**Fin de la Section 1.** On a `gold_dev`, `gold_test`, `textes_dev`, `textes_test`, `ids_dev`, `ids_test`. La Section 2 les utilisera pour la baseline spaCy.